# 05 Final ML model and submission file

**Decision (taken before opening the holdout):** blend (geometric average) of **ridge** and **XGBoost**, both with
- **our label**: `log(value / mean curve)`; prediction = mean curve x exp(model output). The mean curve is the geometric mean of the training batches per day;
- the 14 base features **plus the `next_*` feeds** (feed dosed right after each sample; allowed because test feed schedules are given. It reflects what the operator saw, so it would *not* be available at day 0 in real life: say so in the presentation);
- XGBoost with Philip's settings.

**Steps**
1. CV on the 39 dev batches (frozen folds) -> the score we log.
2. **Holdout check, once:** fit on dev, score the 10 holdout batches. Compared with the per-day mean curve on the same batches. No changes are made after seeing it.
3. Refit on all 49 labelled batches, predict the 12 test batches.
4. Write the submission in the exact template format and check it.
5. Plot Batch54 (the only test batch outside the training range) by eye.

Nothing is submitted by this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from xgboost import XGBRegressor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs" / "ml"
OUT.mkdir(parents=True, exist_ok=True)
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]
FEEDS = ["feed_substrate", "feed_b", "feed_c", "feed_d"]

## Table and split

In [ ]:
def build_table(df):
    rows = []
    for b, g in df.groupby("batch_id"):
        g = g.sort_values("elapsed_time")
        samples = g[g.volume.notna()]                  # sample rows (test batches: volume only)
        d0 = samples.iloc[0]
        feeds = g[g[FEEDS].notna().any(axis=1)]
        times = samples.elapsed_time.values
        for i in range(1, len(samples)):
            s = samples.iloc[i]
            t = s.elapsed_time
            t_next = times[i + 1] if i + 1 < len(times) else np.inf
            before = feeds[feeds.elapsed_time < t]
            last_day = before[before.elapsed_time >= t - 1]
            after = feeds[(feeds.elapsed_time >= t) & (feeds.elapsed_time < t_next)]
            r = dict(batch_id=b, elapsed_time=t, day=t, vcd0=d0.vcd, viability0=d0.vcd / d0.tcd,
                     substrate0=d0.substrate, byproduct0=d0.byproduct, volume0=d0.volume)
            for f in FEEDS:
                r[f"cum_{f}"] = before[f].sum()
                r[f"last24h_{f}"] = last_day[f].sum()
                r[f"next_{f}"] = after[f].sum()
            for c in TARGETS:
                r[c] = s[c]
            rows.append(r)
    return pd.DataFrame(rows)


tab = build_table(pd.read_csv(ROOT / "data" / "bioreaction_remaining.csv"))
splits = pd.read_csv(ROOT / "outputs" / "splits.csv")
tab = tab.merge(splits[["batch_id", "role", "fold"]], on="batch_id", how="left")
tab["dayi"] = tab.elapsed_time.round().astype(int)

FEATURES = ["day", "vcd0", "viability0", "substrate0", "byproduct0", "volume0"] + \
           [f"{p}_{f}" for f in FEEDS for p in ("cum", "last24h", "next")]
dev = tab[tab.role == "dev"].reset_index(drop=True)
holdout = tab[tab.role == "holdout"].reset_index(drop=True)
test = tab[tab.role == "test"].reset_index(drop=True)
print(len(FEATURES), "features |", dev.batch_id.nunique(), "dev,", holdout.batch_id.nunique(), "holdout,",
      test.batch_id.nunique(), "test batches")
assert (len(dev), len(holdout), len(test)) == (507, 130, 156)
assert test[TARGETS].isna().all().all() and not tab[FEATURES].isna().any().any()

## Model

In [ ]:
def mean_curve(train, t):
    m = np.exp(np.log(train[TARGETS]).groupby(train.dayi).mean())
    return np.column_stack([np.interp(t, m.index.values, m[c].values) for c in TARGETS])


def make(kind):
    if kind == "ridge":
        return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 30)))
    return XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=3, min_child_weight=3,
                        subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
                        n_jobs=4, random_state=0)


def fit_predict(train, new):
    """Blend of ridge and XGBoost on our label. Everything is fit on `train` only."""
    btr, bnew = mean_curve(train, train.elapsed_time), mean_curve(train, new.elapsed_time)
    out = []
    for j, c in enumerate(TARGETS):
        y = np.log(train[c].values / btr[:, j])
        mu = np.mean([make(k).fit(train[FEATURES], y).predict(new[FEATURES]) for k in ("ridge", "xgb")], axis=0)
        out.append(bnew[:, j] * np.exp(mu))
    return np.column_stack(out)


def mape(obs, pred):
    s = 100 * np.mean(np.abs((obs - pred) / obs), axis=0)
    return dict(zip(TARGETS + ["average"], np.append(s, s.mean()).round(2)))

## 1. Cross-validation on dev (the score we log)

In [ ]:
rows_m, rows_f = [], []
for k in sorted(dev.fold.unique()):
    tr, va = dev[dev.fold != k].reset_index(drop=True), dev[dev.fold == k].reset_index(drop=True)
    rows_f.append(mape(va[TARGETS].values, fit_predict(tr, va)))
    rows_m.append(mape(va[TARGETS].values, mean_curve(tr, va.elapsed_time)))
cv_final, cv_mean = pd.DataFrame(rows_f), pd.DataFrame(rows_m)
cv = pd.DataFrame({"final blend (mean)": cv_final.mean(), "final blend (std)": cv_final.std(),
                   "mean curve (mean)": cv_mean.mean(), "mean curve (std)": cv_mean.std()}).round(2)
cv

## 2. Holdout check (opened once)
Fit on the 39 dev batches only, score the 10 holdout batches.

In [ ]:
pred_h = fit_predict(dev, holdout)
hold = pd.DataFrame({"final blend": mape(holdout[TARGETS].values, pred_h),
                     "mean curve": mape(holdout[TARGETS].values, mean_curve(dev, holdout.elapsed_time))})
hold

## 3. Refit on all 49 labelled batches and predict the 12 test batches

In [ ]:
labelled = pd.concat([dev, holdout]).reset_index(drop=True)
assert labelled.batch_id.nunique() == 49
pred_t = fit_predict(labelled, test)
pt = test[["batch_id", "elapsed_time"]].copy()
pt[TARGETS] = pred_t
print(pt.shape)

## 4. Submission in the exact template format
Each template row is matched to the nearest sample time of the same batch. The file is only written if every check passes.

In [ ]:
tpl = pd.read_csv(ROOT / "data" / "bioreaction_template.csv")
rows, used, gap = [], set(), 0.0
for _, r in tpl.iterrows():
    q = pt[pt.batch_id == r.batch_id]
    i = (q.elapsed_time - r.elapsed_time).abs().idxmin()
    gap = max(gap, abs(q.loc[i, "elapsed_time"] - r.elapsed_time))
    used.add(i)
    rows.append([r.batch_id, r.elapsed_time, *q.loc[i, TARGETS]])
sub = pd.DataFrame(rows, columns=tpl.columns)

assert list(sub.columns) == list(tpl.columns)
assert len(sub) == 156 == len(tpl), len(sub)
assert not sub[TARGETS].isna().any().any(), "NaN found"
assert (sub[TARGETS] > 0).all().all(), "non-positive value"
assert (sub.batch_id == tpl.batch_id).all() and np.allclose(sub.elapsed_time, tpl.elapsed_time)
assert len(used) == 156, "a prediction was used twice"
print("all checks passed | largest time mismatch to the template:", round(gap, 3), "days")
sub.to_csv(OUT / "submission_ml_blend.csv", index=False)
print("written:", OUT / "submission_ml_blend.csv")

## 5. Batch54 by eye
Batch54 is the only test batch outside the training range (very low day-0 substrate). Grey = the dev batches, black = the mean curve, red = our prediction for Batch54.

In [ ]:
b54 = sub[sub.batch_id == "Batch54"]
d0 = tab[tab.batch_id == "Batch54"].iloc[0]
print("Batch54 day-0: vcd0 %.2f, substrate0 %.2f, byproduct0 %.2f (dev range of substrate0: %.2f-%.2f)" %
      (d0.vcd0, d0.substrate0, d0.byproduct0, dev.substrate0.min(), dev.substrate0.max()))
mc = np.exp(np.log(labelled[TARGETS]).groupby(labelled.dayi).mean())
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for a, c in zip(ax.ravel(), TARGETS):
    for _, g in labelled.groupby("batch_id"):
        a.plot(g.elapsed_time, g[c], color="0.85", lw=0.6)
    a.plot(mc.index, mc[c], "k", lw=2, label="mean curve")
    a.plot(b54.elapsed_time, b54[c], "o-", color="tab:red", label="Batch54 prediction")
    a.set_title(c); a.set_xlabel("day")
ax[0, 0].legend()
plt.tight_layout()
plt.savefig(OUT / "batch54.png", dpi=110)
plt.show()